# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/Users/lukevalerio/miniconda3/bin/python


In [2]:
# NOTE (setup): This cell originally ran
#   pip install tensorflow-model-optimization==0.8.0
# On this machine the "tinyml-arduino" (miniconda) kernel already provides a
# compatible TF-MOT build. Pinning 0.8.0 would downgrade/break the working stack,
# and the assignment says not to install/uninstall packages here, so the install
# is intentionally disabled.
import tensorflow_model_optimization as tfmot
print("tensorflow-model-optimization already available:", tfmot.__version__)

tensorflow-model-optimization already available: 0.8.1


In [3]:
# NOTE (setup): This cell originally ran
#   pip install keras==2.14.0
# Disabled for the same reason: this kernel already uses a compatible Keras
# (via the tf_keras legacy backend), and forcing keras==2.14.0 would break
# TensorFlow 2.21. Install intentionally skipped.
import tensorflow as tf
print("Keras backend in use via tf.keras:", tf.keras.__name__)

Keras backend in use via tf.keras: tf_keras.api._v2.keras


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.21.0
TF-MOT version: 0.8.1



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

X = df[feature_names].values
y = df["Class"].values

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique labels:", np.unique(y))

X shape: (178, 13)
y shape: (178,)
Unique labels: [0 1 2]


In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

Train samples: 124
Test samples: 54


In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled train mean (~0):", np.round(X_train_scaled.mean(axis=0)[:3], 4))
print("Scaled train std  (~1):", np.round(X_train_scaled.std(axis=0)[:3], 4))

Scaled train mean (~0): [ 0. -0.  0.]
Scaled train std  (~1): [1. 1. 1.]


In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape:", y_test_cat.shape)

y_train_cat shape: (124, 3)
y_test_cat shape: (54, 3)


In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax'),
])

model.summary()

Model: "sequential"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 dense (Dense)               (None, 64)                896       


 dense_1 (Dense)             (None, 32)                2080      


 dense_2 (Dense)             (None, 3)                 99        


Total params: 3075 (12.01 KB)


Trainable params: 3075 (12.01 KB)


Non-trainable params: 0 (0.00 Byte)


_________________________________________________________________


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train_scaled, y_train_cat,
    epochs=20, batch_size=8, validation_split=0.2, verbose=2
)

Epoch 1/20


13/13 - 0s - loss: 0.9361 - accuracy: 0.6162 - val_loss: 0.7571 - val_accuracy: 0.8400 - 401ms/epoch - 31ms/step


Epoch 2/20


13/13 - 0s - loss: 0.6953 - accuracy: 0.8990 - val_loss: 0.5598 - val_accuracy: 0.9200 - 42ms/epoch - 3ms/step


Epoch 3/20


...

13/13 - 0s - loss: 0.0224 - accuracy: 1.0000 - val_loss: 0.0335 - val_accuracy: 1.0000 - 24ms/epoch - 2ms/step


Epoch 19/20


13/13 - 0s - loss: 0.0197 - accuracy: 1.0000 - val_loss: 0.0293 - val_accuracy: 1.0000 - 23ms/epoch - 2ms/step


Epoch 20/20


13/13 - 0s - loss: 0.0178 - accuracy: 1.0000 - val_loss: 0.0275 - val_accuracy: 1.0000 - 23ms/epoch - 2ms/step


In [12]:
# Step 7: Evaluate the model and print:
# - Training accuracy and Test accuracy (part (a) asks for both)
# - Classification report
# - Confusion matrix

train_loss, train_acc = model.evaluate(X_train_scaled, y_train_cat, verbose=0)
test_loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f"Training accuracy: {train_acc:.4f}")
print(f"Test accuracy:     {test_acc:.4f}")

y_true = np.argmax(y_test_cat, axis=1)
y_pred = np.argmax(model.predict(X_test_scaled, verbose=0), axis=1)

print("\nClassification report:")
print(classification_report(y_true, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

Training accuracy: 1.0000
Test accuracy:     1.0000



Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

import os

def file_size_kb(path):
    """Return the size of a file in kilobytes."""
    return os.path.getsize(path) / 1024.0

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_base_model = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_base_model)

print(f"Base TFLite model size: {file_size_kb('model_base.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmphai4ba8s/assets


INFO:tensorflow:Assets written to: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmphai4ba8s/assets


Base TFLite model size: 14.08 KB


W0000 00:00:1785383205.587493 60384263 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785383205.587511 60384263 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1785383205.587825 60384263 reader.cc:83] Reading SavedModel from: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmphai4ba8s
I0000 00:00:1785383205.588598 60384263 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1785383205.588603 60384263 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmphai4ba8s
I0000 00:00:1785383205.594222 60384263 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1785383205.595171 60384263 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1785383205.633301 60384263 loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmphai4ba8s
I0000 00:00:1785383205.643848 60

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    X_test is expected to already be preprocessed (scaled) the same way as training.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) default optimizations, (b) representative dataset,
        # (c) INT8-only builtin ops, (d) int8 input/output.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.
    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    y_pred = []
    for i in range(len(X_test)):
        x = X_test[i:i + 1].astype(np.float32)

        # If the input is quantized, quantize the sample with its scale/zero-point.
        if input_details['dtype'] in (np.int8, np.uint8):
            in_scale, in_zp = input_details['quantization']
            x = np.round(x / in_scale + in_zp).astype(input_details['dtype'])

        interpreter.set_tensor(input_details['index'], x)
        interpreter.invoke()
        out = interpreter.get_tensor(output_details['index'])[0]

        # If the output is quantized, dequantize it back to float.
        if output_details['dtype'] in (np.int8, np.uint8):
            out_scale, out_zp = output_details['quantization']
            out = (out.astype(np.float32) - out_zp) * out_scale

        y_pred.append(np.argmax(out))

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")
    print(classification_report(y_true, y_pred))
    print(confusion_matrix(y_true, y_pred))

In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8'     saved as 'model_int8.tflite'
# - 'float16'  saved as 'model_float16.tflite'
# - 'dynamic'  saved as 'model_dynamic.tflite'

quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'int8', 'model_int8.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'float16', 'model_float16.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'dynamic', 'model_dynamic.tflite')

INFO:tensorflow:Assets written to: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmpmku_ylpb/assets


INFO:tensorflow:Assets written to: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmpmku_ylpb/assets


/Users/lukevalerio/miniconda3/lib/python3.13/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1785383206.310638 60384263 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785383206.310653 60384263 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1785383206.310821 60384263 reader.cc:83] Reading SavedModel from: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmpmku_ylpb
I0000 00:00:1785383206.311621 60384263 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1785383206.311627 60384263 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmpmku_ylpb
I0000 00:00:1785383206.319543 60384263 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1785383206.362200 60384263 loader.cc:220] Running initialization op on SavedModel bundle at pat


INT8 TFLite model size: 8.00 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


INFO:tensorflow:Assets written to: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmpjdv6cm8j/assets


...


FLOAT16 TFLite model size: 8.95 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


INFO:tensorflow:Assets written to: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmpkva__sh8/assets


INFO:tensorflow:Assets written to: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmpkva__sh8/assets



DYNAMIC TFLite model size: 8.55 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


W0000 00:00:1785383208.277111 60384263 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785383208.277130 60384263 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1785383208.277285 60384263 reader.cc:83] Reading SavedModel from: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmpkva__sh8
I0000 00:00:1785383208.278091 60384263 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1785383208.278096 60384263 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmpkva__sh8
I0000 00:00:1785383208.284296 60384263 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1785383208.322067 60384263 loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmpkva__sh8
I0000 00:00:1785383208.335906 60384263 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 58625 microseconds.
/Users/lukeva

## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# - initial_sparsity = 0.5, final_sparsity = 0.7
# - end_step approx = training_samples / batch_size * epochs

batch_size = 8
epochs = 10
num_train_samples = int(X_train_scaled.shape[0] * 0.8)   # validation_split=0.2 is held out
end_step = int(np.ceil(num_train_samples / batch_size) * epochs)
print("Pruning end_step:", end_step)

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step,
)

Pruning end_step: 130


In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu) / Dense(32, relu) / Dense(num_classes, softmax)
# Each Dense layer is wrapped with prune_low_magnitude().

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = Sequential([
    tf.keras.layers.Input(shape=(num_features,)),
    prune_low_magnitude(Dense(64, activation='relu'), pruning_schedule=pruning_schedule),
    prune_low_magnitude(Dense(32, activation='relu'), pruning_schedule=pruning_schedule),
    prune_low_magnitude(Dense(num_classes, activation='softmax'), pruning_schedule=pruning_schedule),
])

pruned_model.summary()

Model: "sequential_1"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 prune_low_magnitude_dense_  (None, 64)                1730      


 3 (PruneLowMagnitude)                                           


...

 5 (PruneLowMagnitude)                                           


Total params: 6057 (23.67 KB)


Trainable params: 3075 (12.01 KB)


Non-trainable params: 2982 (11.66 KB)


_________________________________________________________________


In [18]:
# Step 3: Compile with categorical_crossentropy and accuracy.
# - Train for 10 epochs, batch_size=8, validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to callbacks

pruned_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

pruned_model.fit(
    X_train_scaled, y_train_cat,
    epochs=10, batch_size=8, validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],
    verbose=2,
)

Epoch 1/10


13/13 - 1s - loss: 1.0542 - accuracy: 0.4545 - val_loss: 0.8642 - val_accuracy: 0.6000 - 1s/epoch - 101ms/step


Epoch 2/10


13/13 - 0s - loss: 0.6796 - accuracy: 0.7576 - val_loss: 0.5984 - val_accuracy: 0.8800 - 28ms/epoch - 2ms/step


Epoch 3/10


...

13/13 - 0s - loss: 0.1968 - accuracy: 0.9899 - val_loss: 0.4201 - val_accuracy: 0.9200 - 26ms/epoch - 2ms/step


Epoch 9/10


13/13 - 0s - loss: 0.3460 - accuracy: 0.9798 - val_loss: 0.3759 - val_accuracy: 0.9200 - 28ms/epoch - 2ms/step


Epoch 10/10


13/13 - 0s - loss: 0.2907 - accuracy: 0.9899 - val_loss: 0.3321 - val_accuracy: 0.9200 - 28ms/epoch - 2ms/step


In [19]:
# Step 4: Strip pruning wrappers, convert to TFLite, save "model_pruned.tflite", print size.

stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)
tflite_pruned_model = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_pruned_model)

print(f"Pruned TFLite model size: {file_size_kb('model_pruned.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmp02ao2exh/assets


INFO:tensorflow:Assets written to: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmp02ao2exh/assets


W0000 00:00:1785383210.797460 60384263 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785383210.797474 60384263 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.


Pruned TFLite model size: 14.11 KB


I0000 00:00:1785383210.797625 60384263 reader.cc:83] Reading SavedModel from: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmp02ao2exh
I0000 00:00:1785383210.798212 60384263 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1785383210.798216 60384263 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmp02ao2exh
I0000 00:00:1785383210.801565 60384263 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1785383210.816438 60384263 loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmp02ao2exh
I0000 00:00:1785383210.822166 60384263 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 24543 microseconds.


In [20]:
# Step 5: Evaluate using the stripped model.

y_true = np.argmax(y_test_cat, axis=1)
y_pred = np.argmax(stripped_pruned_model.predict(X_test_scaled, verbose=0), axis=1)

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.95      0.97        19
           1       0.95      1.00      0.98        21
           2       1.00      1.00      1.00        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

[[18  1  0]
 [ 0 21  0]
 [ 0  0 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu) / Dense(16, relu) / Dense(num_classes, softmax)

student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax'),
])

student_model.summary()

Model: "sequential_2"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 dense_6 (Dense)             (None, 32)                448       


 dense_7 (Dense)             (None, 16)                528       


 dense_8 (Dense)             (None, 3)                 51        


Total params: 1027 (4.01 KB)


Trainable params: 1027 (4.01 KB)


Non-trainable params: 0 (0.00 Byte)


_________________________________________________________________


In [22]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels.

teacher_preds_soft = model.predict(X_train_scaled, verbose=0)
print("Teacher soft labels shape:", teacher_preds_soft.shape)

Teacher soft labels shape: (124, 3)


In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1.
# (b) Define distillation_loss() that splits the combined labels and blends two
#     categorical_crossentropy losses with alpha = 0.5.

y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)
print("Combined label shape:", y_train_combined.shape)


def distillation_loss(y_true_combined, y_pred):
    # First num_classes columns are the hard labels, the rest are the soft labels.
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]

    alpha = 0.5
    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)
    return alpha * hard_loss + (1.0 - alpha) * soft_loss

Combined label shape: (124, 6)


In [24]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss.
# - Train for 10 epochs, batch_size=8, validation_split=0.2
# (No accuracy metric here: the combined label has 2*num_classes columns, which
#  does not match the 3-class prediction shape a built-in metric would expect.)

student_model.compile(optimizer='adam', loss=distillation_loss)

student_model.fit(
    X_train_scaled, y_train_combined,
    epochs=10, batch_size=8, validation_split=0.2, verbose=2
)

Epoch 1/10


13/13 - 0s - loss: 1.1896 - val_loss: 1.0821 - 470ms/epoch - 36ms/step


Epoch 2/10


13/13 - 0s - loss: 1.0145 - val_loss: 0.9368 - 33ms/epoch - 3ms/step


Epoch 3/10


...

13/13 - 0s - loss: 0.3215 - val_loss: 0.3397 - 30ms/epoch - 2ms/step


Epoch 9/10


13/13 - 0s - loss: 0.2659 - val_loss: 0.2844 - 25ms/epoch - 2ms/step


Epoch 10/10


13/13 - 0s - loss: 0.2191 - val_loss: 0.2415 - 33ms/epoch - 3ms/step


In [25]:
# Step 5: Convert the student model to TFLite, save "model_kd.tflite", print size.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd_model = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_kd_model)

print(f"KD student TFLite model size: {file_size_kb('model_kd.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmp2z0ph3_8/assets


INFO:tensorflow:Assets written to: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmp2z0ph3_8/assets


KD student TFLite model size: 6.12 KB


W0000 00:00:1785383212.510930 60384263 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785383212.510943 60384263 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1785383212.511090 60384263 reader.cc:83] Reading SavedModel from: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmp2z0ph3_8
I0000 00:00:1785383212.512059 60384263 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1785383212.512065 60384263 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmp2z0ph3_8
I0000 00:00:1785383212.518301 60384263 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1785383212.553543 60384263 loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmp2z0ph3_8
I0000 00:00:1785383212.564667 60384263 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 53580 microseconds.


In [26]:
# Step 6: Predictions on X_test_scaled, then classification report + confusion matrix.

y_true = np.argmax(y_test_cat, axis=1)
y_pred = np.argmax(student_model.predict(X_test_scaled, verbose=0), axis=1)

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [27]:
# Part (e): Combine techniques for further size reduction.
# Strategy: take the smallest architecture (the distilled STUDENT model) and apply
# full INT8 quantization to it. This stacks knowledge distillation (fewer, smaller
# weights) with int8 quantization (1 byte/weight) for the smallest model while keeping
# accuracy high. The written justification belongs in the [Dis] PDF.

quantize_and_evaluate(student_model, X_test_scaled, y_test_cat, 'int8', 'model_kd_int8.tflite')

print("\n--- Model size comparison (KB) ---")
for name in ["model_base.tflite", "model_dynamic.tflite", "model_float16.tflite",
             "model_int8.tflite", "model_pruned.tflite", "model_kd.tflite",
             "model_kd_int8.tflite"]:
    if os.path.exists(name):
        print(f"{name:24} {file_size_kb(name):.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmplyet0u_5/assets


INFO:tensorflow:Assets written to: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmplyet0u_5/assets



INT8 TFLite model size: 4.77 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]

--- Model size comparison (KB) ---
model_base.tflite        14.08 KB
model_dynamic.tflite     8.55 KB
model_float16.tflite     8.95 KB
model_int8.tflite        8.00 KB
model_pruned.tflite      14.11 KB
model_kd.tflite          6.12 KB
model_kd_int8.tflite     4.77 KB


/Users/lukevalerio/miniconda3/lib/python3.13/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1785383213.315426 60384263 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785383213.315441 60384263 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1785383213.315586 60384263 reader.cc:83] Reading SavedModel from: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmplyet0u_5
I0000 00:00:1785383213.316349 60384263 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1785383213.316353 60384263 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/_6/dlf_px9j4jl_nlqtql_xp0jr0000gq/T/tmplyet0u_5
I0000 00:00:1785383213.322497 60384263 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1785383213.359794 60384263 loader.cc:220] Running initialization op on SavedModel bundle at pat

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
